In [0]:
#1 — Load Production Models
import mlflow.pyfunc

purchase_model = mlflow.pyfunc.load_model(
"models:/workspace.default.ecommerce_purchase_model@production"
)

fraud_model = mlflow.pyfunc.load_model(
"models:/workspace.default.ecommerce_fraud_model@production"
)

In [0]:
#2 — Load Gold Dataset
df = spark.read.table(
"workspace.ecommerce_lakehouse.gold_ml_training_dataset"
)

display(df)

In [0]:
#Cast Feature Columns
from pyspark.sql.functions import col

for c in feature_cols:
    df = df.withColumn(c, col(c).cast("double"))

In [0]:
#3 — Define Feature Columns
feature_cols = [
"total_events",
"views",
"add_to_cart",
"purchases",
"avg_price",
"frequency",
"monetary_value",
"recency_days"
]

In [0]:
#verify feature schema
df.select(feature_cols).printSchema()


In [0]:
#4 — Create Vectorized Pandas UDF (Purchase Prediction).. This allows distributed prediction across Spark workers.
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType
import pandas as pd

@pandas_udf(DoubleType())
def predict_purchase_udf(*cols):

    pdf = pd.concat(cols, axis=1)
    pdf.columns = feature_cols

    preds = purchase_model.predict(pdf)

    return pd.Series(preds)

In [0]:
#5 — Fraud Prediction UDF
@pandas_udf(DoubleType())
def predict_fraud_udf(*cols):

    pdf = pd.concat(cols, axis=1)
    pdf.columns = feature_cols

    preds = fraud_model.predict(pdf)

    return pd.Series(preds)

In [0]:
#6 — Apply Distributed Scoring..Spark scores data in parallel
scored_df = df.withColumn(
"purchase_prediction",
predict_purchase_udf(*[df[c] for c in feature_cols])
)

In [0]:
# Add fraud prediction..Now the dataset contains model predictions.
scored_df = scored_df.withColumn(
"fraud_prediction",
predict_fraud_udf(*[df[c] for c in feature_cols])
)

In [0]:
#7 — Save Predictions Table. Create final inference dataset.
scored_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable(
"workspace.ecommerce_lakehouse.gold_predictions"
)

In [0]:
#8 — Verify Predictions
display(
spark.read.table(
"workspace.ecommerce_lakehouse.gold_predictions"
))

In [0]:
#9 — Optimize Table,This improves query performance for dashboards.
spark.sql("""
OPTIMIZE workspace.ecommerce_lakehouse.gold_predictions
""")